# QC/trimming samples 
- (sequenced Jan 2024)
- PSTR, OFAV, OANN, MCAV, MMEA
- 2019 and 2022 

In [ ]:
# unzip files, make sample list, rename files in dir

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs012024/unziplist%j.out  # %j = job ID

# unzip files
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw

# unzip 
gunzip *.fastq.gz

## Get sample list 
ls *.fastq* | sed -E 's/_S[0-9]+_R[12]_001\.fastq.*//' | sort -u > /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/sampleids.txt

# remove sequencer ID from actual file names 
for file in *; do
    new_name=$(echo "$file" | sed 's/_S[0-9]\+//')
    mv "$file" "$new_name"
done

# job id- 51476024

In [ ]:
# qc

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-qc-%j.out  # %j = job ID

module load conda/latest
conda activate qc

# Define the paths and variables
FILEPATH='/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw'
OUTPUT_RESULTS='/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed' 
NSLOTS=4  
SAMPLE_NAMES_FILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/sampleids.txt"

cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw
# Check if the file exists
if [ ! -e "$SAMPLE_NAMES_FILE" ]; then
    echo "Error: $SAMPLE_NAMES_FILE does not exist."
    exit 1
fi

# Read each line from the file and perform actions
while IFS= read -r sample_id; do
    # Form the full file names
    input_r1="$FILEPATH/${sample_id}_R1_001.fastq"
    input_r2="$FILEPATH/${sample_id}_R2_001.fastq"
    
    # Ensure the input files exist before running the tools
    if [ ! -e "$input_r1" ] || [ ! -e "$input_r2" ]; then
        echo "Error: Input files do not exist for sample $sample_id"
        continue
    fi

    # Run trim_galore
    trim_galore -j "$NSLOTS" -q 20 --phred33 --length 20 --paired $input_r1 $input_r2 --fastqc -o $OUTPUT_RESULTS --dont_gzip

done < "$SAMPLE_NAMES_FILE"


# JOB-ID: 51535798
# script file: /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024
#trimmed read seqs in folder: /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed
# (raw seqs also in scratch workspace for now & lab hard drive. will transition to zipped in project)

In [ ]:
# ugh ran out of time limit so have to run again 
# will try to pick up where i left off ...and give more time

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/slurm-qc-%j.out  # %j = job ID

### QC
# file to restart on: 	062019_BEL_CBC_T1_21_PAST_R1_001_trimmed.fq
module load conda/latest
conda activate qc

# Define the paths and variables
FILEPATH='/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw'
OUTPUT_RESULTS='/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed' 
NSLOTS=8  
SAMPLE_NAMES_FILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/sampleids.txt"

cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw
while IFS= read -r sample_id; do
    input_r1="$FILEPATH/${sample_id}_R1_001.fastq"
    input_r2="$FILEPATH/${sample_id}_R2_001.fastq"

    # Check if both F&R have already been run 
    if [ -e "$OUTPUT_RESULTS/${sample_id}_R1_001_val_1.fq" ]; then
        echo "Skipping $sample_id - output already exists."
        continue
    fi

    trim_galore -j "$NSLOTS" -q 20 --phred33 --length 20 --paired "$input_r1" "$input_r2" --fastqc -o "$OUTPUT_RESULTS" --dont_gzip
done < "$SAMPLE_NAMES_FILE"

# run multiqc (summary of fastqc) 
conda deactivate
conda activate multiqc
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed
multiqc .
    
# calculate read depth of raw and trimmed files 
# create reads count file 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
echo "sample,read_count,step">reads.csv

# raw reads 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw
# this will read both F & R ... leaving just as sanity check but i can remove later 
for FILE in *.fastq; do
    NAME=$(basename "$FILE")
    COUNT=$(( $(wc -l < "$FILE") / 4 ))
    echo "$NAME,$COUNT,raw" >> ../reads.csv
done

# trimmed reads 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed
for FILE in *_val_1.fq; do
    NAME=$(basename "$FILE" _val_1.fq)
    COUNT=$(( $(wc -l < "$FILE") / 4 ))
    echo "$NAME,$COUNT,trimmed" >> ../reads.csv
done


# JOB-ID:51564657
# script name qc2

In [ ]:
# OMG ran out of time again - trying with --qos=long?
# same script as above with changes in sbatch:

#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 96:00:00  # Job time limit - extended?
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-qc3-%j.out  # %j = job ID


# should start after: 122022_BEL_CBC_T3_132_MCAV_R1_001_val_1.fq



# JOB ID: 51612558
# script name qc3